# 1. LIBRARIES

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to sys.path to allow imports from functions_final.py
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib as mpl
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import joblib

# Import functions from functions_final.py
from functions_final import *
from UQpy.distributions import Uniform, JointIndependent

# Set matplotlib parameters for better aesthetics
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False,
                        'figure.dpi': 100
                    })

# 2. LOADING THE CARBONATION MODEL

### 2.1 Load

In [2]:
# Load the best model
name_best_model = r'C:\Users\Renata Pensin\Desktop\ic_victor\beam_problem_1\model_NeuralNetwork_MLP_fold_4.pkl'

# Load the model
model = joblib.load(name_best_model)
print("Carbonation model loaded successfully!")
print(f"   Expected features: {model.feature_names_in_}")

Carbonation model loaded successfully!
   Expected features: ['CO2 (%)' 'fc (MPa)' 'RH (%)' 'Type of cement' 'Exposure conditions'
 't (years)']


### 2.2 Testing model

In [3]:
mat = {
        'f_ck [MPa]': 30,
        'Type of cement': 2
      }
expo = {
          'Installation year': 1990,
          'Exposure conditions': 2,
          'Relative humidity [%]': 40
        }
geo = {'cover[mm]':30}
load = {}
predictor = CO2Predictor()
beam_with_rh = Beam(geo=geo, mat=mat, load=load, expo=expo)
predictor.set_beam(beam_with_rh)
profile = predictor.carbonation_profile(model_=model, lifetime=150)

### 2.3 Carbonation profile over time

In [4]:
profile

,calendar year,t (years),CO2 (%),carbonation depth (mm)
0,1990,0,0.03574,0.589410
1,2000,10,0.03690,10.735841
2,2010,20,0.03893,15.324331
3,2020,30,0.04132,18.963275
4,2030,40,0.04407,21.881557
5,2040,50,0.04718,24.647107
6,2050,60,0.05065,27.325433
7,2060,70,0.05448,29.712950
8,2070,80,0.05867,31.815637
9,2080,90,0.06322,33.933596


### 2.4 Test carbonation depth at 2025

In [5]:
carb_depth_mm = predictor.carbonation_depth_at_time(profile, 2025)
carb_depth_mm

20.42241603415687

# 3. DEFINITION OF RANDOM VARIABLES

Defines the probability distributions for the input variables:
- Concrete compressive strength ($f_{ck}$);
- Relative humidity (RH).
- Cover depth (cov).

### 3.1 Design variables

In [6]:
fck_min = 20 # MPa
fck_max = 50 # MPa
rh_min  = 50 # %
rh_max  = 80 # %
cov_min = 20  # mm
cov_max = 60  # mm

### 3.2 Fixed parameters for the analysis

In [7]:
cement_type          = 3
installation_year    = 1990
exposure_conditions  = 2
n_samples            = 100        # Number of design samples. Use 1 for testing one sample
n_latent_samples     = 1000       # Number of latent samples per design sample
n_samples_validation = 3          # Number of validation samples
n_lambdas            = 4          # Number of λs to be predicted (λ1, λ2, λ3, λ4)

### 3.3 Samples

In [8]:
# Distributions of random variables
fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist   = Uniform(loc=cov_min, scale=cov_max - cov_min)

# Joint distribution
joint = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

# Generate samples
x_pce_rvs = joint.rvs(n_samples)

# Report sample statistics
print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations: {n_samples * n_latent_samples}")
print("\nSample statistics:")
print(f"   fck:      {x_pce_rvs[:, 0].min():.1f} - {x_pce_rvs[:, 0].max():.1f} MPa (mean: {x_pce_rvs[:, 0].mean():.1f} MPa)")
print(f"   RH:       {x_pce_rvs[:, 1].min():.1f} - {x_pce_rvs[:, 1].max():.1f}% (mean: {x_pce_rvs[:, 1].mean():.1f}%)")
print(f"   cover:    {x_pce_rvs[:, 2].min():.1f} - {x_pce_rvs[:, 2].max():.1f} mm (mean: {x_pce_rvs[:, 2].mean():.1f} mm)")

Samples generated successfully!
   Number of design samples: 100
   Number of latent samples per design sample: 1000
   Total simulations: 100000

Sample statistics:
   fck:      20.0 - 50.0 MPa (mean: 35.5 MPa)
   RH:       50.9 - 80.0% (mean: 65.1%)
   cover:    20.1 - 59.9 mm (mean: 39.5 mm)


# 4. EVALUATION OF CARBONATION PROGRESS USING GENERALIZED LAMBDA DISTRIBUTION

### 4.1 Step time

In [9]:
# times               = np.arange(0, 150, 20)
times               = [10, 20, 30, 40]
all_df_full = []
all_df_unique = []

### 4.2 Loop over design samples

In [10]:
print("="*60)
print("BUILDING THE DURABILITY EMULATOR")
print("="*60)

for t in times:
    # ============================================================
    # EMULATOR FUNCTION - CARBONATION DEPTH AND LAMBDAS
    # ============================================================
    print(f'\n{"-"*40}')
    print(f'PROCESSING EMULATOR FOR TIME STEP: {t} years')
    print(f'{"-"*40}')
    
    df_full, df_unique = emulator_function_time_durability(
                                                                x=x_pce_rvs,
                                                                names_x_variables=["fck", "rh", "cov"],
                                                                carb_model=model,
                                                                cement_type=cement_type,
                                                                installation_year=installation_year,
                                                                exposure_conditions=exposure_conditions,
                                                                time_step=t,
                                                                n_latent_samples=n_latent_samples,
                                                                verbose=False
                                                            )
    

    df_full['time'] = t
    df_unique['time'] = t
    
    # Store the datasets for this time step in the lists for later merging
    all_df_full.append(df_full)
    all_df_unique.append(df_unique)
    
    print(f'1. The dataset for time {t} has been generated and stored in memory!')
    
    # =============================================================
    # BUILDING THE PCE METAMODEL
    # =============================================================
    lambda_cols      = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']
    y_pce_rvs        = df_unique[lambda_cols].to_numpy()
    max_degree       = 3
    polynomial_basis = TotalDegreeBasis(joint, max_degree)
    least_squares    = LeastSquareRegression()
    pce_metamodel    = PolynomialChaosExpansion(polynomial_basis=polynomial_basis, regression_method=least_squares)                                                                                                        
    
    # Train
    pce_metamodel.fit(x_pce_rvs, y_pce_rvs)
    
    # Save the PCE metamodel for this time step
    filename = f'pce_metamodel_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
    with open(filename, 'wb') as f:
        dill.dump(pce_metamodel, f)
    print(f'2. PCE training dataset has been saved!')
    
    # =============================================================
    # VALIDATION THE PCE METAMODEL
    # =============================================================
    x_pce_rvs_val = joint.rvs(n_samples_validation)
    df_full_val, df_unique_val = emulator_function_time_durability(
                                                                        x=x_pce_rvs_val,
                                                                        names_x_variables=["fck", "rh", "cov"],
                                                                        carb_model=model,
                                                                        cement_type=cement_type,
                                                                        installation_year=installation_year,
                                                                        exposure_conditions=exposure_conditions,
                                                                        time_step=t,
                                                                        n_latent_samples=n_latent_samples,
                                                                        verbose=False
                                                                    )
    lambda_cols      = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']
    y_pce_val_true   = df_unique_val[lambda_cols].to_numpy()
    y_pce_val_pred   = pce_metamodel.predict(x_pce_rvs_val)
    mse_por_lambda   = []
    r2_por_lambda    = []
    
    for ii in range(n_lambdas):
        verdade = y_pce_val_true[:, ii]
        predito = y_pce_val_pred[:, ii]
        # MSE computing
        mse = mean_squared_error(verdade, predito)
        mse_por_lambda.append(mse)
        # R² computing
        r2 = r2_score(verdade, predito)
        r2_por_lambda.append(r2)
        
    statistics_ = pd.DataFrame({'MSE λ1': mse_por_lambda[0], 'MSE λ2': mse_por_lambda[1], 'MSE λ3': mse_por_lambda[2], 'MSE λ4': mse_por_lambda[3], 'R² λ1': r2_por_lambda[0], 'R² λ2': r2_por_lambda[1], 'R² λ3': r2_por_lambda[2], 'R² λ4': r2_por_lambda[3]}, index=[0])
    filename_stats = f'pce_validation_stats_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
    with open(filename_stats, 'wb') as f:
        dill.dump(statistics_, f)
    print(f'3. PCE statistcs has been saved!')

# ============================================================
# SAVING THE COMBINED DATASET
# ============================================================
print(f'\n{"="*60}')
print("MERGING ALL DATASETS")
print(f'{"="*60}')

# Junta todos os dataframes em um só (ignora os index originais para evitar duplicações)
combined_df_full = pd.concat(all_df_full, ignore_index=True)
combined_df_unique = pd.concat(all_df_unique, ignore_index=True)

# Salva os datasets unificados
filename_combined_full = f'dataset_full_COMBINED_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
with open(filename_combined_full, 'wb') as f:
    dill.dump(combined_df_full, f)

filename_combined_unique = f'dataset_unique_COMBINED_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
with open(filename_combined_unique, 'wb') as f:
    dill.dump(combined_df_unique, f)

print(f"Combined 'full' dataset saved with shape: {combined_df_full.shape}")
print(f"Combined 'unique' dataset saved with shape: {combined_df_unique.shape}")

BUILDING THE DURABILITY EMULATOR

----------------------------------------
PROCESSING EMULATOR FOR TIME STEP: 10 years
----------------------------------------
1. The dataset for time 10 has been generated and stored in memory!
2. PCE training dataset has been saved!
3. PCE statistcs has been saved!

----------------------------------------
PROCESSING EMULATOR FOR TIME STEP: 20 years
----------------------------------------
1. The dataset for time 20 has been generated and stored in memory!
2. PCE training dataset has been saved!
3. PCE statistcs has been saved!

----------------------------------------
PROCESSING EMULATOR FOR TIME STEP: 30 years
----------------------------------------
1. The dataset for time 30 has been generated and stored in memory!
2. PCE training dataset has been saved!
3. PCE statistcs has been saved!

----------------------------------------
PROCESSING EMULATOR FOR TIME STEP: 40 years
----------------------------------------
1. The dataset for time 40 has been 

In [11]:
filename_stats = f'dataset_unique_COMBINED_install_1990_cement_3_exposure_2.pkl'
with open(filename_stats, 'rb') as f:
    file_loaded = dill.load(f)
file_loaded

,fck,rh,cov,lambda 1,lambda 2,lambda 3,lambda 4,time
0,29.157554,59.200283,26.231141,15.864735,0.666926,0.136255,0.194104,10
1,24.753314,69.550967,56.791837,44.810857,0.458280,0.096625,0.066129,10
2,34.650896,79.758856,57.709501,52.273708,0.465369,0.146392,0.139110,10
3,39.838488,76.756710,22.780044,18.013427,1.015052,0.090470,0.136551,10
4,29.777220,61.523996,48.862160,38.808077,0.458335,0.169645,0.138442,10
...,...,...,...,...,...,...,...,...
395,49.562477,71.520335,29.780199,22.271385,0.710265,0.089915,0.151331,40
396,48.103542,72.522459,55.189470,47.169284,0.462673,0.126389,0.114720,40
397,39.605418,62.200575,58.819367,46.425365,0.405206,0.138630,0.179171,40
398,44.900992,54.439318,49.609129,39.240570,0.453217,0.161712,0.192576,40


In [ ]:
# 1. DATA PREPARATION 

features = ['fck', 'rh', 'cov', 'time']
targets  = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']

X = combined_df_unique[features].to_numpy()
y = combined_df_unique[targets].to_numpy()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

print("="*60)
print(f"SPLIT FINISHED: {len(X_train)} training samples | {len(X_val)} validation samples")
print("="*60 + "\n")

# 2. TRAINING NEURAL NETWORKS (MLP) - COMPARISON

mlp_config = {
    'hidden_layer_sizes': (64, 64),
    'max_iter': 2000,
    'random_state': 42,
    'early_stopping': True
}

print("4 Independent Neural Networks")

modelos_independentes = []
y_pred_val_indep = np.zeros_like(y_val)

for i, col in enumerate(targets):
    modelo_individual = make_pipeline(
        StandardScaler(),
        MLPRegressor(**mlp_config)
    )
    
    modelo_individual.fit(X_train, y_train[:, i])
    modelos_independentes.append(modelo_individual)
    y_pred_val_indep[:, i] = modelo_individual.predict(X_val)
    
    r2 = r2_score(y_val[:, i], y_pred_val_indep[:, i])
    mse = mean_squared_error(y_val[:, i], y_pred_val_indep[:, i])
    print(f"   {col} -> R² : {r2:.4f} | MSE: {mse:.4f}")

print("1 Global Neural Network (Multiple Outputs)")

mlp_config_multi = mlp_config.copy()
mlp_config_multi['hidden_layer_sizes'] = (100, 100) 

modelo_global = make_pipeline(
    StandardScaler(),
    MLPRegressor(**mlp_config_multi)
)

modelo_global.fit(X_train, y_train)
y_pred_val_global = modelo_global.predict(X_val)

for i, col in enumerate(targets):
    r2 = r2_score(y_val[:, i], y_pred_val_global[:, i])
    mse = mean_squared_error(y_val[:, i], y_pred_val_global[:, i])
    print(f"   {col} -> R² : {r2:.4f} | MSE: {mse:.4f}")


SPLIT CONCLUÍDO: 320 amostras de treino | 80 de validação

ABORDAGEM 1: 4 Redes Neurais Independentes
   lambda 1 -> R² : 0.9939 | MSE: 1.1291
   lambda 2 -> R² : 0.9379 | MSE: 0.0016
   lambda 3 -> R² : 0.5501 | MSE: 0.0020
   lambda 4 -> R² : 0.1850 | MSE: 0.0020

ABORDAGEM 2: 1 Rede Neural Global (Múltiplas Saídas)
   lambda 1 -> R² : 0.9655 | MSE: 6.3460
   lambda 2 -> R² : 0.1255 | MSE: 0.0221
   lambda 3 -> R² : 0.1556 | MSE: 0.0037
   lambda 4 -> R² : -3.3893 | MSE: 0.0108

RESUMO DA COMPARAÇÃO CONCLUÍDO!


In [12]:
# 80/20 SPLIT (Train and Test)

# Assuming combined_df_unique is already loaded
features = ['fck', 'rh', 'cov', 'time']
targets  = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']

X = combined_df_unique[features].to_numpy()
y = combined_df_unique[targets].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("="*60)
print("DATA SPLIT COMPLETED:")
print(f" -> Training and Internal Validation: {len(X_train)} samples (80%)")
print(f" -> Unseen Test for Evaluation      : {len(X_test)} samples (20%)")
print("="*60 + "\n")

# Training for 4 Independent Neural Networks
mlp_config = {
    'hidden_layer_sizes': (64, 64),
    'max_iter': 2000,
    'random_state': 42,
    'early_stopping': True,
    'validation_fraction': 0.20 
}

independent_models = []
y_pred_test = np.zeros_like(y_test)

print("Training the Final Neural Networks...\n")
for i, col in enumerate(targets):
    # The pipeline standardizes and trains
    individual_model = make_pipeline(StandardScaler(), MLPRegressor(**mlp_config))
    
    # Train with Training data (validation happens automatically inside)
    individual_model.fit(X_train, y_train[:, i])
    independent_models.append(individual_model)
    
    y_pred_test[:, i] = individual_model.predict(X_test)
    
    r2_test = r2_score(y_test[:, i], y_pred_test[:, i])
    mse_test = mean_squared_error(y_test[:, i], y_pred_test[:, i])
    print(f"Results on the TEST set for {col}:")
    print(f"   R² : {r2_test:.4f}  |  MSE: {mse_test:.4f}")

# Save the final models to a file
joblib.dump(independent_models, 'pipeline_mlp_final.pkl')
print("\nFinal models saved as 'pipeline_mlp_final.pkl'")

DATA SPLIT COMPLETED:
 -> Training and Internal Validation: 320 samples (80%)
 -> Unseen Test for Evaluation      : 80 samples (20%)

Training the Final Neural Networks...

Results on the TEST set for lambda 1:
   R² : 0.9916  |  MSE: 2.1563
Results on the TEST set for lambda 2:
   R² : 0.9003  |  MSE: 0.0026
Results on the TEST set for lambda 3:
   R² : 0.3612  |  MSE: 0.0028
Results on the TEST set for lambda 4:
   R² : 0.5239  |  MSE: 0.0034

Final models saved as 'pipeline_mlp_final.pkl'


In [13]:
# Inference Pipeline Function
def predict_lambda_evolution(fck, rh, cov, time_list, trained_models):
    """
    Receives project parameters and a list of time steps.
    Returns a Numpy matrix (N_times x 4) with the values of λ1, λ2, λ3 and λ4.
    """
    results_matrix = []
    
    for t in time_list:
        # Set up the current scenario 
        current_scenario = np.array([[fck, rh, cov, t]])
        lambdas_this_year = []
        
        # Pass the scenario through the 4 experts
        for model in trained_models:
            # Get the first value from the array returned by the prediction
            prediction = model.predict(current_scenario)[0]
            lambdas_this_year.append(prediction)
            
        results_matrix.append(lambdas_this_year)
        
    return np.array(results_matrix)

# Testing the pipeline function
print("Testing the Pipeline Function")

time_analyse = [10, 20, 30, 40, 50] 
fck_proj = 30
rh_proj  = 65
cov_proj = 40

# Run the pipeline
lambda_matrix = predict_lambda_evolution(fck_proj, rh_proj, cov_proj, time_analyse, independent_models)

print(f"Results for beam with fck={fck_proj}, rh={rh_proj}%, cov={cov_proj}mm:")
print("Columns: [λ1, λ2, λ3, λ4] | Rows: Years [10, 20, 30, 40, 50]\n")
print(np.round(lambda_matrix, 4))

Testing the Pipeline Function
Results for beam with fck=30, rh=65%, cov=40mm:
Columns: [λ1, λ2, λ3, λ4] | Rows: Years [10, 20, 30, 40, 50]

[[2.98886e+01 6.14300e-01 7.82000e-02 1.11000e-01]
 [2.52779e+01 5.09600e-01 1.67000e-02 1.27000e-01]
 [2.11973e+01 4.56500e-01 4.80000e-02 1.96200e-01]
 [1.99342e+01 4.43200e-01 5.87000e-02 2.02100e-01]
 [1.93357e+01 3.99600e-01 7.68000e-02 2.51000e-01]]
